# Lab 4: An LLM Extraction You Can Trust

---

## What this lab is

In Lab 3 you built emissions numbers from plant-level data. Many numbers a climate-finance team needs only exist as **text** in sustainability reports, so analysts increasingly use an LLM to turn report text into a table.

This lab treats the LLM as a **measurement instrument**, exactly like the utilisation rates and emission factors of Lab 3: it has an error rate, the error rate has an **interval**, some errors are random and some are **systematic**, and a risk committee needs to know which is which.

* **Part A. From text to table.** Define a schema, parse LLM output, and label a few excerpts yourself.
* **Part B. Measure the instrument.** Accuracy per field, an interval on accuracy, and what errors do to a sector total.
* **Part C. Self-consistency.** Ask the model 5 times. Does disagreement tell you where it is wrong?
* **Part D. Retrieval.** A mini-RAG: can we even find the right paragraph?
* **Part E. Communicate.** Should the bank adopt this pipeline?

## Learning goals

1. Turn free text into **validated structured data** (JSON + pydantic).
2. Evaluate an LLM against a **gold standard** and put an **interval** on its accuracy.
3. Distinguish **random** LLM errors (caught by resampling) from **systematic** ones (not caught).
4. Design a **human-review policy** from an uncertainty signal and a cost.
5. Explain why **retrieval** is a separate source of error in RAG.

## Conventions

* Questions are **[QUAL]** (2-4 sentences) or **[QUANT]** (a number from your code plus one sentence).
* Write each answer in the cell right under the question.
* Units: emissions in **tCO2e** (tonnes), production in **Mt** (million tonnes) of crude steel.
* Boxes marked **[OPTIONAL]** can be skipped.
* Run **Kernel -> Restart & Run All** before submitting.

## About the data

All 20 companies are **fictional**, but the excerpts copy the way real reports are written (different units, fiscal years, combined scopes, missing values). `recorded_responses.json` contains LLM answers **recorded in advance** (1 answer at temperature 0 and 5 answers at temperature 0.7 per excerpt) so that everyone works on the same outputs and no API key is needed. If you have access to an LLM, the optional box in A2 lets you run your own and compare.

## Submission

1. Full name: Labergere Adrien   Student ID: B00822082

**Group / repo name:** `aidams-lab4-labergere`
**Submitter (one person):** Labergere Adrien
**Repo URL:** _(to fill in after pushing to GitHub)_

### What to submit

- This notebook (`lab4.ipynb`) with all tasks completed and cells run (outputs visible)
- A short `README.md` with how to run the notebook

Create a GitHub repo, push the notebook and README, invite the instructor, and paste the **repo URL** above.

**Send submission info to my email (1 email per group).** Include the GitHub repo URL.

**Done when:** all names are filled in and the notebook contains outputs (no need for the instructor to re-run it).

## Setup (given)

In [1]:
import json, re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pydantic import BaseModel, ValidationError

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

DATA = Path(".")  # the json files sit next to the notebook, not in a data/ subfolder
excerpts = pd.DataFrame(json.loads((DATA / "excerpts.json").read_text()))
gold = (pd.DataFrame(json.loads((DATA / "gold_labels.json").read_text())).set_index("id")
          .astype({"report_year": "Int64", "net_zero_year": "Int64"}))
recorded = pd.DataFrame(json.loads((DATA / "recorded_responses.json").read_text()))
corpus = json.loads((DATA / "aurelia_report_chunks.json").read_text())

FIELDS = ["company", "report_year", "scope1_tco2", "scope2_tco2", "production_mt", "net_zero_year"]
SEED = 42
print(excerpts.shape, gold.shape, recorded.shape, recorded.kind.value_counts().to_dict())

(20, 2) (20, 6) (120, 6) {'sample': 100, 'greedy': 20}


### The prompt and the LLM connection (given)

`ask_llm(prompt, temperature)` returns the model's raw text. With `BACKEND = "recorded"` (default) you do not need it: the answers are already in `recorded`. The other backends are there for the optional live run.

In [2]:
PROMPT = """You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
{text}
"""

BACKEND = "recorded"          # "recorded" | "anthropic" | "openai" | "ollama"
MODEL = None                  # e.g. "claude-haiku-5-5" (anthropic), your course model (openai), "llama3.2" (ollama)


def ask_llm(prompt: str, temperature: float = 0.0) -> str:
    if BACKEND == "anthropic":
        import anthropic                                   # pip install anthropic; needs ANTHROPIC_API_KEY
        msg = anthropic.Anthropic().messages.create(
            model=MODEL, max_tokens=400, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return msg.content[0].text
    if BACKEND == "openai":
        from openai import OpenAI                          # pip install openai; needs OPENAI_API_KEY
        r = OpenAI().chat.completions.create(
            model=MODEL, temperature=temperature,
            messages=[{"role": "user", "content": prompt}])
        return r.choices[0].message.content
    if BACKEND == "ollama":
        import requests                                    # needs a local `ollama serve`
        r = requests.post("http://localhost:11434/api/generate", timeout=120, json={
            "model": MODEL, "prompt": prompt, "stream": False, "options": {"temperature": temperature}})
        return r.json()["response"]
    raise RuntimeError("BACKEND is 'recorded': use the `recorded` table instead of calling the model.")


print(PROMPT.format(text=excerpts.text[0]))

You extract climate data from a steel company's report excerpt.
Return ONLY a JSON object with these keys:
  company        (string)
  report_year    (integer, the year the figures refer to; for a fiscal year use the calendar year it starts in)
  scope1_tco2    (number in tonnes CO2e, or null if not reported separately)
  scope2_tco2    (number in tonnes CO2e, market-based if both are given, or null if not reported separately)
  production_mt  (crude steel production in million metric tonnes, or null)
  net_zero_year  (integer year of a net zero / carbon neutrality target, or null if there is none)
  confidence     (number between 0 and 1: how confident you are that every field is correct)
Convert all units. Never compute a value that is not reported.

Excerpt:
Aurelia Steel - Annual Climate Report 2024. In 2024 Aurelia Steel produced 12.6 million tonnes of crude steel across its three integrated sites. Gross Scope 1 emissions were 24,190,000 tCO2e, and Scope 2 emissions (market-based)

---
# PART A. From text to table

## A0. Be the annotator first

> **Pause & think.** No code. Do this **before** you look at `gold`.

### Q1 [QUAL] Label three excerpts yourself

Print the excerpts **E03**, **E09** and **E19** and fill in the six fields for each by hand, following the rules in `PROMPT`. Then compare with `gold.loc[["E03", "E09", "E19"]]`. Which field was hardest for you, and why? If two people in your group disagreed, what does that say about the "truth" we will score the LLM against?

In [3]:
for eid in ["E03", "E09", "E19"]:
    print(eid, "-", excerpts.set_index("id").loc[eid, "text"])
    print()

E03 - Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 million tonnes of crude steel. Greenhouse gas emissions (unit: 10,000 tonnes CO2): direct emissions (Scope 1) 4,108; indirect emissions from purchased electricity (Scope 2) 512. The Group will strive to peak carbon emissions before 2030 and achieve carbon neutrality by 2060.

E09 - Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Scope 2 emissions amounted to 8.7 MtCO2e, a 4% decrease year on year. Ostend Metals is committed to net zero emissions by 2050.

E19 - Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of raw steel at our DRI-EAF facilities. Scope 1 emissions: 5.1 million metric tonnes CO2e; Scope 2: 410,000 metric tonnes CO2e. Bluewater targets net zero by 2050.



**Your answer (Q1):**

Before looking at gold, here's what I got by hand:

- **E03**: Lianhe, 2023, scope1 = 41,080,000 (4,108 x 10,000 - easy to miss that unit), scope2 = 5,120,000, production 21.4 Mt, net zero 2060.
- **E09**: Ostend Metals, 2024, scope1/scope2 = null (only a combined number is given, so I didn't try to split it), production 4.4 Mt, net zero 2050.
- **E19**: Bluewater Iron, 2024, scope1 = 5,100,000, scope2 = 410,000, production 2.99 Mt (converted from 3.3 million short tons), net zero 2050.

All three match `gold` exactly. The hardest one was E09 - it's tempting to put the combined 8.7 Mt figure *somewhere* instead of leaving two fields blank, but the excerpt just doesn't break it down. E19's short-ton conversion is also easy to skip if you're reading fast. If a groupmate had done these independently, I think E09 is where we'd disagree, since nothing in the text forces you to leave it null - that's really the point of the gold standard here: it's not "the truth from the report," it's the answer a careful reader agrees on once you apply the "don't guess" rule consistently.

## A1. A schema and a parser

**Goal.** Create

* a pydantic model `SteelDisclosure` with the six fields of the prompt plus `confidence` (types: `str`, `int`, `Optional[float]`, `Optional[int]` as appropriate);
* a function `parse_response(raw)` that finds the JSON object inside the raw text, validates it with `SteelDisclosure`, and returns `(record, None)` on success or `(None, error_message)` on failure;
* a DataFrame `pred` with one row per **successfully parsed** response (columns `id`, `kind`, `sample` and the fields) and a list `failures`.

> **Hint.** Models often wrap JSON in text or code fences: `re.search(r"\{.*\}", raw, flags=re.S)`. Then `SteelDisclosure.model_validate_json(...)` and `record.model_dump()`. Catch `ValidationError`.

In [4]:
class SteelDisclosure(BaseModel):
    company: str
    report_year: int
    scope1_tco2: Optional[float] = None
    scope2_tco2: Optional[float] = None
    production_mt: Optional[float] = None
    net_zero_year: Optional[int] = None
    confidence: Optional[float] = None


def parse_response(raw: str):
    match = re.search(r"\{.*\}", raw, flags=re.S)
    if not match:
        return None, "no JSON object found in raw text"
    try:
        record = SteelDisclosure.model_validate_json(match.group(0))
        return record, None
    except ValidationError as e:
        return None, str(e)


rows, failures = [], []
for _, r in recorded.iterrows():
    record, err = parse_response(r["raw_text"])
    if record is None:
        failures.append({"id": r["id"], "kind": r["kind"], "sample": r["sample"],
                          "raw_text": r["raw_text"], "error": err})
    else:
        row = record.model_dump()
        row.update({"id": r["id"], "kind": r["kind"], "sample": r["sample"]})
        rows.append(row)

pred = pd.DataFrame(rows)[["id", "kind", "sample"] + FIELDS + ["confidence"]]
print("pred:", pred.shape, " failures:", len(failures))
pred.head()

pred: (119, 10)  failures: 1


,id,kind,sample,company,report_year,scope1_tco2,scope2_tco2,production_mt,net_zero_year,confidence
0,E01,greedy,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
1,E01,sample,0,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.940
2,E01,sample,1,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.910
3,E01,sample,2,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.930
4,E01,sample,3,Aurelia Steel,2024,"24,190,000.000","1,310,000.000",12.600,"2,050.000",0.960


### Q2 [QUANT + QUAL] What breaks?

(a) How many of the 120 recorded responses fail to parse, and why (look at their `raw_text`)? How many responses needed the regex because they were wrapped in text? (b) Give **two** options for handling a failed response in a production pipeline, and one risk of each. In Part B we will count a failed response as **wrong on every field**: why is that the honest choice for an accuracy figure?

In [5]:
needed_regex = recorded["raw_text"].apply(lambda t: not t.strip().startswith("{"))
print(f"{len(failures)} / {len(recorded)} responses failed to parse")
print(f"{needed_regex.sum()} / {len(recorded)} responses needed the regex (wrapped in text / code fences)")

for f in failures:
    print("\n---", f["id"], f["kind"], "sample", f["sample"], "---")
    print(f["raw_text"])
    print("error:", f["error"].splitlines()[-1] if f["error"] else f["error"])

1 / 120 responses failed to parse
7 / 120 responses needed the regex (wrapped in text / code fences)

--- E11 greedy sample 0 ---
{
  "company": "Anatolia Celik",
  "report_year": 2024,
  "scope1_tco2": 4120000,
  "scope2_tco2": 2350000,
  "production_mt": "8.1 Mt",
  "net_zero_year": 2053,
  "confidence": 0.94
}
error:     For further information visit https://errors.pydantic.dev/2.10/v/float_parsing


**Your answer (Q2):**

(a) Only **1 of 120** responses fails to parse: E11's greedy answer, because it writes `"production_mt": "8.1 Mt"` as a string instead of a number. **7** other responses needed the regex fallback because the model wrapped the JSON in chatty text plus a code fence ("Sure! Here is the extracted data: ..."). Note that 8.1 is the *shipments* figure, not production (9.8), so this answer would have been wrong even if it had parsed. The validation made the problem loud instead of letting it slip into the table.

(b) Two options in production: retry the call (risk: it can fail the same way again if the cause is systematic, and it costs time/money), or send it straight to a human (risk: doesn't scale if failures are common). Counting a parse failure as wrong on every field is the honest choice because the pipeline produced nothing usable for that excerpt - dropping it from the denominator instead would make the accuracy look better than it really is.

> ### [OPTIONAL] A2. Run your own model
>
> If you have an LLM available, set `BACKEND` and `MODEL` above, run your model on **E03, E09, E19** with `temperature=0`, parse the outputs with `parse_response` and compare them with the recorded greedy answers and with `gold`. Did your model make the same mistakes?

In [6]:
# [OPTIONAL] needs a local Ollama server: `ollama pull llama3.2`, then `ollama serve` (leave it running)
BACKEND, MODEL = "ollama", "llama3.2"

def same(a, b):
    """Quick check for this cell only: True if both missing, else 2% tolerance on numbers / exact match."""
    if pd.isna(a) and pd.isna(b):
        return True
    if pd.isna(a) or pd.isna(b):
        return False
    try:
        return abs(float(a) - float(b)) <= 0.02 * abs(float(b))
    except (TypeError, ValueError):
        return str(a).lower() in str(b).lower() or str(b).lower() in str(a).lower()

ids = ["E03", "E09", "E19"]
text_by_id = excerpts.set_index("id")["text"]
own_rows = []
try:
    for eid in ids:
        raw = ask_llm(PROMPT.format(text=text_by_id[eid]), temperature=0)
        rec, err = parse_response(raw)
        if rec is None:
            print(eid, "failed to parse:", err.splitlines()[0])
            continue
        own_rows.append({**rec.model_dump(), "id": eid})
except Exception as e:
    print("Ollama not reachable (is `ollama serve` running?):", e)

if own_rows:
    own = pd.DataFrame(own_rows).set_index("id")
    rec_greedy = pred[pred["kind"] == "greedy"].set_index("id")
    for eid in own.index:
        cmp = pd.DataFrame({"own model": own.loc[eid, FIELDS],
                            "recorded greedy": rec_greedy.loc[eid, FIELDS],
                            "gold": gold.loc[eid, FIELDS]})
        cmp["own correct"] = [same(cmp.loc[f, "own model"], cmp.loc[f, "gold"]) for f in FIELDS]
        print(eid)
        display(cmp)

E03


,own model,recorded greedy,gold,own correct
company,Lianhe Iron & Steel Group,Lianhe Iron & Steel Group,Lianhe Iron & Steel Group,True
report_year,2023,2023,2023,True
scope1_tco2,"4,108.000","4,108,000.000","41,080,000.000",False
scope2_tco2,512.000,"512,000.000","5,120,000.000",False
production_mt,21.400,21.400,21.400,True
net_zero_year,2060,"2,060.000",2060,True


E09


,own model,recorded greedy,gold,own correct
company,Ostend Metals NV,Ostend Metals,Ostend Metals,True
report_year,2024,2024,2024,True
scope1_tco2,NaN,"8,700,000.000",NaN,True
scope2_tco2,8.700,NaN,NaN,False
production_mt,4.400,4.400,4.400,True
net_zero_year,2050,"2,050.000",2050,True


E19


,own model,recorded greedy,gold,own correct
company,Bluewater Iron Corp.,Bluewater Iron,Bluewater Iron,True
report_year,2024,2024,2024,True
scope1_tco2,"5,110,000.000","5,100,000.000","5,100,000.000",True
scope2_tco2,"410,000.000","410,000.000","410,000.000",True
production_mt,3.300,3.300,2.990,False
net_zero_year,2050,"2,050.000",2050,True


**Your answer (A2):**

I ran Llama 3.2 locally with Ollama (temperature 0) on E03, E09 and E19. It made the same kinds of mistakes as the recorded model: on E03 it copies the figures as printed (Scope 1 = 4,108 and Scope 2 = 512) and ignores the "10,000 tonnes" unit, and on E19 it does not convert the short tons (production 3.3 instead of 2.99), exactly like the recorded answer. On E09 it does not leave the scopes empty: it puts the combined 8.7 Mt into Scope 2, without converting it to tonnes, whereas the recorded model put it in Scope 1. It also changes E19's Scope 1 to 5,110,000 instead of 5,100,000, which is still inside the 2% tolerance. Both models get 14 of 18 fields right on these three excerpts, and fail on the same two things, units and a combined figure. That fits the idea of systematic errors, but with only three excerpts it is a hint, not a proof.

---
# PART B. Measure the instrument

## B1. Accuracy per field

**Goal.** Create

* `field_correct(field, pred_value, gold_value)`: True if both are missing; False if only one is missing; for `company` a case- and punctuation-insensitive containment match; for years an exact match; for numbers a **relative tolerance of 2%**;
* `score(df)`: one row per (excerpt, field) with `id`, `field`, `pred`, `gold`, `correct`. Excerpts whose response failed to parse must appear with `correct = False` on every field;
* `greedy_scores = score(...)` for the **greedy** (temperature 0) answers, and a table of accuracy per field.

> **Hint.** Missing values come back as `None` or `NaN`: test with `pd.isna`. To include the failures, reindex the greedy predictions on `gold.index`.

In [7]:
def field_correct(field, pred_value, gold_value):
    p_missing, g_missing = pd.isna(pred_value), pd.isna(gold_value)
    if p_missing and g_missing:
        return True
    if p_missing != g_missing:
        return False
    if field == "company":
        norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
        return norm(gold_value) in norm(pred_value) or norm(pred_value) in norm(gold_value)
    if field in ("report_year", "net_zero_year"):
        return int(pred_value) == int(gold_value)
    if gold_value == 0:
        return float(pred_value) == 0
    return abs(float(pred_value) - float(gold_value)) <= 0.02 * abs(float(gold_value))


def score(df_preds):
    """One row per (excerpt, field). Excerpts with no successfully parsed row score False everywhere."""
    by_id = {eid: g.iloc[0] for eid, g in df_preds.groupby("id")}
    records = []
    for eid in gold.index:
        row = by_id.get(eid)
        for field in FIELDS:
            gold_v = gold.loc[eid, field]
            pred_v = row[field] if row is not None else None
            records.append({"id": eid, "field": field, "pred": pred_v, "gold": gold_v,
                             "correct": field_correct(field, pred_v, gold_v)})
    return pd.DataFrame(records)


greedy_pred = pred[pred["kind"] == "greedy"]
greedy_scores = score(greedy_pred)
acc_per_field = greedy_scores.groupby("field").correct.mean().sort_values().to_frame("accuracy")
acc_per_field

,accuracy
field,
scope1_tco2,0.750
net_zero_year,0.850
scope2_tco2,0.850
production_mt,0.900
report_year,0.900
company,0.950


### Q3 [QUANT + QUAL] Where and how does it fail?

(a) Give the overall accuracy and the **worst field**. (b) Print every wrong (excerpt, field) with `pred` and `gold`, read the excerpt, and sort the errors into **error types** (e.g. unit conversion, wrong year in a table, wrong scope, a value invented for a missing field, parse failure). Which type is most frequent? (c) Which of these errors would **you** also have been likely to make in Q1?

In [8]:
overall_acc = greedy_scores.correct.mean()
worst_field = acc_per_field.index[0]
print(f"Overall accuracy: {overall_acc:.1%}  ({greedy_scores.correct.sum()}/{len(greedy_scores)} fields)")
worst_acc = acc_per_field.loc[worst_field, "accuracy"]
print(f"Worst field: {worst_field} ({worst_acc:.1%})")

wrong = greedy_scores[~greedy_scores.correct].merge(
    excerpts.rename(columns={"text": "excerpt_text"}), on="id")
wrong[["id", "field", "pred", "gold", "excerpt_text"]]

Overall accuracy: 86.7%  (104/120 fields)
Worst field: scope1_tco2 (75.0%)


,id,field,pred,gold,excerpt_text
0,E03,scope1_tco2,"4,108,000.000","41,080,000.000","Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 millio..."
1,E03,scope2_tco2,"512,000.000","5,120,000.000","Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 millio..."
2,E05,scope1_tco2,"42,042,000.000",NaN,Tamarind Steel Ltd - Business Responsibility and Sustainability Report FY2024. Crude steel production rose 6% to 18....
3,E07,scope1_tco2,"14,200,000.000","13,800,000.000",Great Lakes Ironworks Inc. - 2024 Sustainability Report. Key performance data: Scope 1 emissions (MtCO2e) 2022: 14.9...
4,E08,report_year,2025,2024,"Hokuriku Special Steel Co., Ltd. - Integrated Report 2025. Results for FY2024 (April 2024 to March 2025): crude stee..."
5,E09,scope1_tco2,"8,700,000.000",NaN,Ostend Metals NV - Sustainability Statement 2024 (ESRS E1). Crude steel production: 4.4 Mt. Combined Scope 1 and Sco...
6,E10,net_zero_year,"2,030.000",<NA>,Sahara Steel Holdings - Sustainability Report 2024. The group produced 5.6 million tonnes of crude steel through its...
7,E11,company,None,Anatolia Celik,Anatolia Celik A.S. - 2024 Sustainability Report. Liquid steel production: 9.8 million tonnes; shipments: 8.1 millio...
8,E11,report_year,None,2024,Anatolia Celik A.S. - 2024 Sustainability Report. Liquid steel production: 9.8 million tonnes; shipments: 8.1 millio...
9,E11,scope1_tco2,None,"4,120,000.000",Anatolia Celik A.S. - 2024 Sustainability Report. Liquid steel production: 9.8 million tonnes; shipments: 8.1 millio...


**Your answer (Q3):**

(a) Overall accuracy is **86.7%** (104/120). Worst field: **scope1_tco2** at 75%.

(b) The 16 wrong fields fall into four types:
- **Value where null was expected** (5 fields, 5 excerpts): E05 computes Scope 1 from the intensity, E09 puts the combined Scope 1+2 total into Scope 1, E10 and E20 take an unrelated year (a 2030 intensity target, a 2026 plant start-up) as the net-zero year, and E18 writes 0 for "will be disclosed in 2025".
- **Unit conversion** (3 fields, 2 excerpts): E03 misses the "x10,000 tonnes" on both scopes, E19 does not convert short tons.
- **Wrong year** (2 fields): E07 takes the 2023 column of the table, E08 uses the report year (2025) instead of the start of the fiscal year (2024).
- **Parse failure** (6 fields, 1 excerpt): E11.

By number of fields the parse failure is the largest block, but it is a single excerpt. Across excerpts, the most frequent type is a value given where the answer should be null, which is exactly what the prompt says not to do.

(c) In Q1 I got E03 and E19 right, but only because I read slowly and knew the units were the trap, so under time pressure I could easily make those mistakes. E08 (fiscal year vs report year) is probably the other one I could have slipped on. I would not have invented values, since "never compute what isn't reported" was the rule I was holding onto.

## B2. An interval on accuracy

"91% accurate" on 120 fields is an **estimate** from a small sample, just like the utilisation rate in Lab 3. Two ways to put a range on it:

1. **Bayesian (Beta-Binomial).** With a uniform prior Beta(1, 1) and *k* correct out of *n*, the posterior is **Beta(1 + k, 1 + n - k)**. Its 5th and 95th percentiles give a 90% credible interval.
2. **Bootstrap over excerpts.** Resample **excerpts** (not fields) with replacement, recompute accuracy, repeat 2,000 times, take the 5th and 95th percentiles.

**Goal.** Create `k`, `n`, `beta_ci` and `boot_ci` (both 90%).

> **Hint.** `stats.beta(a, b).ppf([0.05, 0.95])`. For the bootstrap, first compute per excerpt the number correct and the number of fields (`groupby("id").correct.agg(["sum", "count"])`), then use the same `rng.integers(...)` trick as Lab 3.

In [9]:
k = int(greedy_scores.correct.sum())
n = len(greedy_scores)
beta_ci = stats.beta(1 + k, 1 + n - k).ppf([0.05, 0.95])

rng = np.random.default_rng(SEED)
per_excerpt = greedy_scores.groupby("id").correct.agg(["sum", "count"])
sums, counts = per_excerpt["sum"].to_numpy(), per_excerpt["count"].to_numpy()
n_boot = 2000
boot_acc = np.empty(n_boot)
for b in range(n_boot):
    pick = rng.integers(0, len(sums), len(sums))
    boot_acc[b] = sums[pick].sum() / counts[pick].sum()
boot_ci = np.percentile(boot_acc, [5, 95])

print(f"k={k}, n={n}, point estimate={k/n:.1%}")
print(f"Beta-Binomial 90% CI: [{beta_ci[0]:.1%}, {beta_ci[1]:.1%}]")
print(f"Bootstrap-over-excerpts 90% CI: [{boot_ci[0]:.1%}, {boot_ci[1]:.1%}]")

k=104, n=120, point estimate=86.7%
Beta-Binomial 90% CI: [80.6%, 90.8%]
Bootstrap-over-excerpts 90% CI: [77.5%, 93.3%]


### Q4 [QUANT + QUAL] How sure are we about "accurate"?

(a) Report both intervals. Which is wider, and why? (Hint: are the six fields of one excerpt independent draws? Think of E11.) (b) Roughly how many excerpts would you need to label to **halve** the width of the interval? (c) A vendor advertises "95% extraction accuracy". Is your result consistent with that claim?

**Your answer (Q4):**

(a) Beta gives **[80.6%, 90.8%]**, the bootstrap over excerpts gives a wider **[77.5%, 93.3%]**. That's because the 6 fields of one excerpt aren't independent draws - E11 is the clear case, one parsing slip makes all 6 of its fields wrong at once. Beta counts that as 6 separate bits of bad evidence, the bootstrap correctly treats it as one unlucky excerpt, so it ends up less confident (wider interval).

(b) Width shrinks roughly like 1/sqrt(n), so halving it needs about **4x** more labelled excerpts - from 20 to around 80.

(c) No - even the narrower interval tops out at 90.8%, below the vendor's 95% claim.

## B3. What do the errors do downstream?

A lender wants the **sector total of Scope 1 emissions** and the **median emission intensity** (tCO2 per tonne of steel) of these 20 companies.

**Goal.** Create a DataFrame `downstream` indexed by excerpt with `scope1_llm`, `scope1_gold`, `error_t` (= LLM - gold, in tCO2, treating a missing value as 0 for the total) and print: total Scope 1 according to the LLM and to gold (MtCO2), the **net** and the **absolute** error, and the median intensity (Scope 1 / production, only where both are present) for LLM and gold.

In [10]:
scope1_llm = greedy_pred.set_index("id")["scope1_tco2"].reindex(gold.index)

downstream = pd.DataFrame({
    "scope1_llm": scope1_llm,
    "scope1_gold": gold["scope1_tco2"],
}, index=gold.index)
downstream["error_t"] = downstream["scope1_llm"].fillna(0) - downstream["scope1_gold"].fillna(0)

total_llm_mt = downstream["scope1_llm"].fillna(0).sum() / 1e6
total_gold_mt = downstream["scope1_gold"].fillna(0).sum() / 1e6
net_error_mt = total_llm_mt - total_gold_mt
abs_error_mt = downstream["error_t"].abs().sum() / 1e6

print(f"Sector total Scope 1 - LLM:  {total_llm_mt:,.1f} MtCO2e")
print(f"Sector total Scope 1 - gold: {total_gold_mt:,.1f} MtCO2e")
print(f"Net error:      {net_error_mt:+,.1f} MtCO2e ({net_error_mt/total_gold_mt:+.1%})")
print(f"Absolute error: {abs_error_mt:,.1f} MtCO2e")
print()
print("Top contributors to the absolute error (MtCO2e):")
print((downstream["error_t"].abs() / 1e6).sort_values(ascending=False).head(3))

intensity_llm = (downstream["scope1_llm"] / (gold["production_mt"] * 1e6)).dropna()
intensity_gold = (downstream["scope1_gold"] / (gold["production_mt"] * 1e6)).dropna()
print()
print(f"Median intensity - LLM:  {intensity_llm.median():.3f} tCO2/t")
print(f"Median intensity - gold: {intensity_gold.median():.3f} tCO2/t")

Sector total Scope 1 - LLM:  252.1 MtCO2e
Sector total Scope 1 - gold: 242.1 MtCO2e
Net error:      +10.1 MtCO2e (+4.2%)
Absolute error: 92.2 MtCO2e

Top contributors to the absolute error (MtCO2e):
id
E05   42.042
E03   36.972
E09    8.700
Name: error_t, dtype: float64

Median intensity - LLM:  1.895 tCO2/t
Median intensity - gold: 1.865 tCO2/t


### Q5 [QUANT + QUAL] Accuracy is a mean, the damage is in the tail

(a) Report the net and absolute error on the sector total. Which **two** excerpts contribute most? (b) Why can a 90% field accuracy hide a large error on a portfolio total, while the **median** intensity barely moves? Connect to Session 4 ("the mean hides the tail"). (c) What would you report to the lender alongside the total?

**Your answer (Q5):**

(a) LLM total is 252.1 MtCO2e vs 242.1 gold, so the **net** error is only +10.1 Mt (+4.2%), but the **absolute** error is 92.2 Mt - much bigger, because E05 (+42 Mt, invented) and E03 (-37 Mt, missed unit) push in opposite directions and mostly cancel out in the net figure. Those two excerpts alone make up about 79 of the 92 Mt.

(b) Field accuracy counts every wrong field the same, whether it is a typo on a small firm or a unit slip on a huge emitter, but a total is dominated by whichever large company is wrong. The median intensity barely moves (1.895 vs 1.865) because it only depends on the middle company, not on how far off an outlier is. It is the same lesson as the mean hiding the tail in Session 4 (VaR vs Expected Shortfall).

(c) I'd give the lender the absolute error too, not just the net number, plus which companies drove most of it (E05, E03), and the median intensity as a sanity check since it is much less sensitive to one bad reading.

---
# PART C. Self-consistency: ask five times

At temperature 0.7 the model **samples** its answer, so asking the same question five times can give different answers. Idea: use the **majority vote** as the answer and the **agreement** (share of samples that agree with the vote) as an uncertainty signal. This is an **ensemble**, like the multi-model comparison of Session 4.

## C1. Vote and agreement

**Goal.** Create `votes`: one row per (excerpt, field) with `vote` (most frequent answer among the 5 samples, missing counted as an answer), `agreement` (count of the vote / 5) and `correct`. Compare the vote's accuracy with the greedy accuracy, and show the accuracy for agreement **5/5**, **4/5** and **3/5 or less**.

> **Hint.** To count values that may be `None`, `NaN` or numbers, turn each into a string key first: `json.dumps(None if pd.isna(v) else v)`.

In [11]:
from collections import Counter

def value_key(v):
    return json.dumps(None if pd.isna(v) else v)

sample_pred = pred[pred["kind"] == "sample"]

vote_rows = []
for eid in gold.index:
    g = sample_pred[sample_pred["id"] == eid]
    for field in FIELDS:
        keys = [value_key(v) for v in g[field]] + [value_key(None)] * (5 - len(g))  # a failed parse counts as "None"
        vote_key, n_agree = Counter(keys).most_common(1)[0]
        vote_rows.append({"id": eid, "field": field, "vote": json.loads(vote_key), "agreement": n_agree / 5})

votes = pd.DataFrame(vote_rows)
votes["gold"] = votes.apply(lambda r: gold.loc[r["id"], r["field"]], axis=1)
votes["correct"] = votes.apply(lambda r: field_correct(r["field"], r["vote"], r["gold"]), axis=1)

print(f"Greedy accuracy: {greedy_scores.correct.mean():.1%}")
print(f"Vote accuracy:   {votes.correct.mean():.1%}")
print()
agreement_bin = pd.cut(votes["agreement"], [0, 0.7, 0.9, 1.01], labels=["<=3/5", "4/5", "5/5"])
votes.groupby(agreement_bin, observed=True).correct.agg(["mean", "count"])

Greedy accuracy: 86.7%
Vote accuracy:   95.0%



,mean,count
agreement,,
<=3/5,0.750,12
4/5,0.933,15
5/5,0.978,93


### Q6 [QUANT + QUAL] Is disagreement a useful signal?

(a) Report the greedy and vote accuracies and the accuracy in each agreement bin. (b) Is low agreement a good warning sign? (c) Compute the accuracy for 5/5 with a Beta(1,1) 90% interval: can you claim it is above 95%?

In [12]:
n5 = votes[votes.agreement == 1.0]
k5, n5n = int(n5.correct.sum()), len(n5)
beta_ci_5 = stats.beta(1 + k5, 1 + n5n - k5).ppf([0.05, 0.95])
print(f"5/5 agreement: {k5}/{n5n} correct = {k5/n5n:.1%}, Beta(1,1) 90% CI = [{beta_ci_5[0]:.1%}, {beta_ci_5[1]:.1%}]")

5/5 agreement: 91/93 correct = 97.8%, Beta(1,1) 90% CI = [93.5%, 99.1%]


**Your answer (Q6):**

(a) Greedy is 86.7%, voting over 5 samples gets to **95%**. By agreement: <=3/5 -> 75% correct, 4/5 -> 93.3%, 5/5 -> 97.8%.

(b) Yes - accuracy climbs pretty cleanly with agreement, so low agreement is a real warning sign, not just noise.

(c) 91/93 correct at 5/5, Beta 90% CI = **[93.5%, 99.1%]**. The lower bound is under 95%, so no, we still can't claim "above 95%" even on the fields the model is unanimous on.

## C2. Confidently wrong

### Q7 [QUAL] The errors that resampling cannot see

Print the (excerpt, field) pairs where **all five samples agree** and the vote is **wrong**. Read the excerpts. (a) What do these errors have in common? (b) Using Session 4 vocabulary, why does asking the same model more times not reveal them? Which fix would (a different prompt, a second model, a unit-conversion rule, a human)? (c) Name the analogue in the Session 4 worked example.

In [13]:
confidently_wrong = votes[(votes.agreement == 1.0) & (~votes.correct)].merge(
    excerpts.rename(columns={"text": "excerpt_text"}), on="id")
confidently_wrong[["id", "field", "vote", "gold", "excerpt_text"]]

,id,field,vote,gold,excerpt_text
0,E03,scope1_tco2,"4,108,000.000","41,080,000.000","Lianhe Iron & Steel Group - Environmental, Social and Governance Report 2023. In 2023 the Group produced 21.4 millio..."
1,E19,production_mt,3.300,2.990,Bluewater Iron Corp. - 2024 Environmental Report. Shipments and production: we produced 3.3 million short tons of ra...


**Your answer (Q7):**

(a) The two unanimous-but-wrong cases are the same unit mistakes as in Q3: E03's Scope 1 (misses the x10,000 every time) and E19's production (never converts the short tons). Same mistake, 5 times in a row.

(b) In Session 4 terms these are systematic errors (structural epistemic uncertainty), not random noise: resampling only shows the spread of the model's randomness, not a bias that all draws share, so 5 samples give 5 identical wrong answers. We need different information: a unit rule (flag excerpts that mention "10,000 tonnes" or "short tons") or a human. A plausibility check on tCO2/t would catch E03 (0.19 instead of 1.92) but not E19 (1.55 instead of 1.71, both plausible).

(c) It is the linear model of the Session 4 worked example: its 95% band missed 3 of 5 later years because the true process was exponential. The model is wrong in the same direction every time, so refitting or resampling inside it never shows it. Only a comparison with the truth (here the gold sample) does.

## C3. The model's own confidence

Each response contains a `confidence` the model wrote about itself.

### Q8 [QUANT + QUAL] Is stated confidence calibrated?

For the **sample** responses, compute the mean stated confidence and the share of responses with **all six fields correct**. Then group responses by confidence (`pd.cut(..., [0, .90, .93, .96, 1])`) and show the share fully correct in each group. Is the model **over-confident**? Is its confidence still useful for **ranking** responses?

In [14]:
def full_correct(row):
    return all(field_correct(f, row[f], gold.loc[row["id"], f]) for f in FIELDS)

sp = sample_pred.copy()
sp["full_correct"] = sp.apply(full_correct, axis=1)

mean_conf = sp["confidence"].mean()
share_full = sp["full_correct"].mean()
print(f"Mean stated confidence:        {mean_conf:.1%}")
print(f"Share fully correct (6/6):     {share_full:.1%}")
print()
conf_bin = pd.cut(sp["confidence"], [0, 0.90, 0.93, 0.96, 1.0])
sp.groupby(conf_bin, observed=True)["full_correct"].agg(["mean", "count"]).rename(columns={"mean": "share_fully_correct"})

Mean stated confidence:        92.8%
Share fully correct (6/6):     55.0%



,share_fully_correct,count
confidence,,
"(0.0, 0.9]",0.286,21
"(0.9, 0.93]",0.457,35
"(0.93, 0.96]",0.737,38
"(0.96, 1.0]",0.833,6


**Your answer (Q8):**

Mean stated confidence is 92.8%, but only 55% of samples are fully correct on all 6 fields - so yes, the model is clearly overconfident. That said, confidence still tracks correctness: the share fully correct goes from about 29% in the lowest confidence bucket up to 83% in the highest, so it's still useful for *ranking* answers, just not as a literal probability of being right.

## C4. A review policy with a price tag

A human analyst can check a field for **EUR 4**. An error that reaches the lender's report costs on average **EUR 150** (restatement, analyst time, reputational risk). Three policies:

| Policy | Rule |
|---|---|
| **P0** | trust the majority vote everywhere |
| **P1** | send to a human every field with agreement **< 5/5** |
| **P2** | send to a human every field with agreement **<= 3/5** |

Assume the human fixes every field they review.

### Q9 [QUANT + QUAL] Which policy?

For each policy compute the number of fields reviewed, the number of errors that remain, and the total cost. Which policy is cheapest? Which errors survive even the best policy, and what one extra check would catch them?

In [15]:
HUMAN_COST, ERROR_COST = 4, 150
policies = {
    "P0 - trust the vote everywhere": lambda a: pd.Series(False, index=a.index),
    "P1 - review if agreement < 5/5": lambda a: a < 1.0,
    "P2 - review if agreement <= 3/5": lambda a: a <= 0.6,
}

rows = []
for name, rule in policies.items():
    reviewed = rule(votes["agreement"])
    errors_remaining = (~votes["correct"]) & (~reviewed)
    cost = reviewed.sum() * HUMAN_COST + errors_remaining.sum() * ERROR_COST
    rows.append({"policy": name, "fields_reviewed": int(reviewed.sum()),
                 "errors_remaining": int(errors_remaining.sum()), "total_cost_eur": cost})

policy_table = pd.DataFrame(rows).set_index("policy")
policy_table

,fields_reviewed,errors_remaining,total_cost_eur
policy,,,
P0 - trust the vote everywhere,0,6,900
P1 - review if agreement < 5/5,27,2,408
P2 - review if agreement <= 3/5,12,3,498


**Your answer (Q9):**

P0 (trust everything): 0 reviewed, 6 errors left, 900 EUR. P1 (review if <5/5): 27 reviewed, 2 errors left, **408 EUR**. P2 (review if <=3/5): 12 reviewed, 3 errors left, 498 EUR. P1 wins. This matches the break-even rule from the lecture (check a field if P(error) > 4/150 = 2.7%): the error rate is 25% at <=3/5 and 6.7% at 4/5, both above it, but only 2.2% at 5/5, below it.

The 2 errors that survive even P1 are the unanimous unit mistakes from Q7 (E03 Scope 1, E19 production), since the model never disagrees with itself on them. The one extra check that would catch them is a unit rule: send to a human any excerpt that uses a non-standard unit (10,000 tonnes, short tons). An intensity plausibility check alone would only catch E03.

---
# PART D. Retrieval: finding the right paragraph

Real reports are 100+ pages, longer than you want to send to a model. **RAG** first **retrieves** the most relevant chunks, then asks the LLM to answer from them. If retrieval misses, the LLM never sees the answer.

`corpus["chunks"]` is a (fictional) report cut into 16 chunks; `corpus["queries"]` are 6 questions with the index of the chunk that answers each one.

## D1. TF-IDF retrieval

**Goal.** Represent chunks and queries as **TF-IDF vectors** (`TfidfVectorizer(stop_words="english")`), score each chunk by **cosine similarity** with the query, and build a table `retrieval` with, per query, the top-3 chunk indices, whether the relevant chunk is **rank 1** (`hit@1`) and whether it is in the **top 3** (`hit@3`).

> **Hint.** `from sklearn.feature_extraction.text import TfidfVectorizer`, `from sklearn.metrics.pairwise import cosine_similarity`; `np.argsort(-scores)[:3]`.

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

chunks = corpus["chunks"]
queries = corpus["queries"]

vec = TfidfVectorizer(stop_words="english")
chunk_vecs = vec.fit_transform(chunks)
query_vecs = vec.transform([q["query"] for q in queries])
sims = cosine_similarity(query_vecs, chunk_vecs)

retr_rows = []
for i, q in enumerate(queries):
    top3 = np.argsort(-sims[i])[:3]
    relevant = set(q["relevant"])
    retr_rows.append({
        "qid": q["qid"], "query": q["query"], "top3": list(top3),
        "hit@1": top3[0] in relevant, "hit@3": bool(relevant & set(top3)),
    })

retrieval = pd.DataFrame(retr_rows)
hit1, hit3 = retrieval["hit@1"].mean(), retrieval["hit@3"].mean()
print(f"hit@1: {hit1:.1%}   hit@3: {hit3:.1%}")
retrieval

hit@1: 66.7%   hit@3: 66.7%


,qid,query,top3,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,"[1, 5, 12]",False,False
1,Q2,net zero target year,"[5, 0, 13]",True,True
2,Q3,crude steel production 2024,"[2, 3, 0]",True,True
3,Q4,investment in hydrogen DRI capex,"[6, 0, 2]",True,True
4,Q5,impact of the carbon price on EBITDA,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,"[3, 6, 1]",False,False


### Q10 [QUANT + QUAL] Why does retrieval miss?

(a) Report hit@1 and hit@3. (b) For each failed query, print the query, the chunk it retrieved first and the relevant chunk. Why did TF-IDF fail? (c) **Embeddings** (dense vectors from a neural model) compare meaning rather than words. Which failure would they most likely fix? (d) In a RAG pipeline, why is a retrieval miss more dangerous than an extraction error?

In [17]:
failed = retrieval[~retrieval["hit@3"]]
for _, row in failed.iterrows():
    relevant_idx = next(q["relevant"][0] for q in queries if q["qid"] == row["qid"])
    qid, query = row["qid"], row["query"]
    print(f"--- {qid}: \"{query}\" ---")
    print("retrieved rank 1:", repr(chunks[row["top3"][0]]))
    print("should have been:", repr(chunks[relevant_idx]))
    print()

--- Q1: "What were Scope 1 emissions in 2024?" ---
retrieved rank 1: 'About this report. Greenhouse gas reporting follows the GHG Protocol. Scope 1 covers emissions from sources we own or control, Scope 2 covers purchased electricity and heat, and Scope 3 covers our value chain.'
should have been: 'Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.'

--- Q6: "flooding exposure of our plants" ---
retrieved rank 1: 'Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.'
should have been: 'Physical risk. Two of our three sites are located on major rivers. An assessment of riverine inundation under high-warming scenarios identified the Linz-Nord site as the most exposed.'



**Your answer (Q10):**

(a) hit@1 = hit@3 = **66.7%** (4/6).

(b) Q1 retrieves the *definition* of Scope 1 instead of the chunk with the number. The right chunk never says "Scope 1" (it says "Direct greenhouse gas emissions ... 24.19 MtCO2e"), whereas the "About this report" chunk repeats "Scope 1" and "emissions", so word matching prefers it. Q6 fails because the query says "flooding" and the report says "riverine inundation": no shared word, even though it is exactly the right content.

(c) Embeddings would fix Q6 pretty easily, since "flooding" and "inundation" are close in meaning (like the flood/inundation example in the lecture). Q1 is less certain: a model that knows Scope 1 means direct emissions could match the right chunk, but the definition chunk is also close in meaning.

(d) A retrieval miss is worse because the model never sees the right text, and it is silent: when the text is silent the model's prior can fill the gap with a plausible number, with no error or null to warn us. With an extraction error at least the right paragraph was in the prompt. (D2 confirms this: Q6 is fixed by embeddings, but Q1 still fails.)

> ### [OPTIONAL] D2. Dense embeddings
>
> `pip install sentence-transformers`, then `SentenceTransformer("all-MiniLM-L6-v2").encode(...)` to embed chunks and queries, and redo D1 with cosine similarity on the embeddings. Does hit@1 improve? Which query still fails?

In [18]:
# pip install sentence-transformers   (first run downloads all-MiniLM-L6-v2)
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
chunk_emb = model.encode(chunks, normalize_embeddings=True)
query_emb = model.encode([q["query"] for q in queries], normalize_embeddings=True)
sims_dense = query_emb @ chunk_emb.T          # unit vectors, so dot product = cosine similarity

dense_rows = []
for i, q in enumerate(queries):
    top3 = np.argsort(-sims_dense[i])[:3]
    relevant = set(q["relevant"])
    dense_rows.append({
        "qid": q["qid"], "query": q["query"], "top3": list(top3),
        "hit@1": top3[0] in relevant, "hit@3": bool(relevant & set(top3)),
    })

retrieval_dense = pd.DataFrame(dense_rows)
print(f"TF-IDF  -> hit@1: {retrieval['hit@1'].mean():.1%}   hit@3: {retrieval['hit@3'].mean():.1%}")
print(f"Dense   -> hit@1: {retrieval_dense['hit@1'].mean():.1%}   hit@3: {retrieval_dense['hit@3'].mean():.1%}")

for _, row in retrieval_dense[~retrieval_dense["hit@1"]].iterrows():
    relevant_idx = next(q["relevant"][0] for q in queries if q["qid"] == row["qid"])
    print(f"\n--- {row['qid']}: \"{row['query']}\" ---")
    print("retrieved rank 1:", repr(chunks[row["top3"][0]]))
    print("should have been:", repr(chunks[relevant_idx]))

retrieval_dense

TF-IDF  -> hit@1: 66.7%   hit@3: 66.7%
Dense   -> hit@1: 83.3%   hit@3: 83.3%

--- Q1: "What were Scope 1 emissions in 2024?" ---
retrieved rank 1: 'Targets. We are committed to reaching net zero greenhouse gas emissions by 2050, with an interim target to reduce Scope 1 and 2 emissions by 35% by 2030 against a 2018 baseline.'
should have been: 'Climate performance. Direct greenhouse gas emissions from our blast furnaces, coke ovens and sinter plants totalled 24.19 MtCO2e in 2024, an intensity of 1.92 tCO2e per tonne of crude steel.'


,qid,query,top3,hit@1,hit@3
0,Q1,What were Scope 1 emissions in 2024?,"[5, 12, 1]",False,False
1,Q2,net zero target year,"[5, 0, 6]",True,True
2,Q3,crude steel production 2024,"[2, 0, 3]",True,True
3,Q4,investment in hydrogen DRI capex,"[6, 8, 7]",True,True
4,Q5,impact of the carbon price on EBITDA,"[8, 13, 7]",True,True
5,Q6,flooding exposure of our plants,"[9, 10, 3]",True,True


**Your answer (D2):**

Dense embeddings improve hit@1 from 66.7% to 83.3% (5/6). Q6 is now fixed: "flooding" and "riverine inundation" share no word but are close in meaning. Q1 still fails, and the right chunk is not even in the top 3: the model retrieves the "Targets" chunk, which talks about Scope 1 and 2 reductions, instead of the chunk with the 24.19 MtCO2e figure. The right chunk never uses the words "Scope 1", so the model matches the topic instead of the number. So embeddings help with synonyms, but not when a chunk about the topic is a better match than the chunk that really contains the answer.

---
# PART E. Communicate

### Q11 [QUAL] Should the bank adopt this pipeline?

Write a note of **about 80 words** to the bank's **Head of ESG Data**, who must decide whether to use this LLM pipeline to build a database of steel-company emissions. Include: the **accuracy with its interval**, the **main failure mode**, your **recommended review policy** and its cost, and **one limitation** of this evaluation. Plain language, no jargon. Use a calibrated word from the IPCC scale at least once.

**Your answer (Q11):**

Hi,

A single run of the model gets about 87% of fields right (90% range: 78-93%), so we cannot claim 95%. Its worst mistakes are unit slips, such as a missing x10,000, repeated identically every time, so asking again does not catch them. I suggest a person checks every field where five repeated runs disagree: EUR 408 on our 120 test fields, versus EUR 900 with no checks. It is likely good enough for a first draft, not for sign-off. Limit: only 20 companies were tested.

Adrien